# Fine-tune Audio Flamingo 3 for Audio Captioning

In this notebook, we'll fine-tune [Audio Flamingo 3](https://huggingface.co/nvidia/audio-flamingo-3-hf) for audio captioning on the [AudioCaps](https://huggingface.co/datasets/OpenSound/AudioCaps) dataset. We cover both full fine-tuning and LoRA-based parameter-efficient training.

Audio Flamingo 3 is NVIDIA's 8B multimodal model for audio understanding and caption generation. Fine-tuning it on audio captioning datasets improves its ability to describe sounds accurately.

**What we'll cover:**
- Loading the AudioCaps dataset in streaming mode
- Setting up a data collator for Audio Flamingo 3's chat template
- Full fine-tuning
- LoRA fine-tuning for parameter-efficient training
- Running inference with the fine-tuned model

In [ ]:
# Install required packages
!pip install -q transformers datasets accelerate torch torchaudio peft bitsandbytes librosa soundfile

## Load Dataset

We use the AudioCaps dataset streamed from Hugging Face, resampled to 16kHz as required by the model's Whisper-based feature extractor.

In [ ]:
from datasets import load_dataset, Audio

dataset = load_dataset("OpenSound/AudioCaps", split="train", streaming=True)
dataset = dataset.shuffle(seed=42, buffer_size=10_000)
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

# Use smaller subsets for demo
train_dataset = dataset.take(100)
eval_dataset = dataset.skip(100).take(20)

## Load Model and Processor

In [ ]:
import torch
from transformers import AudioFlamingo3ForConditionalGeneration, AutoProcessor

model_checkpoint = "nvidia/audio-flamingo-3-hf"

processor = AutoProcessor.from_pretrained(model_checkpoint)
model = AudioFlamingo3ForConditionalGeneration.from_pretrained(
    model_checkpoint,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print(f"Model loaded: {model_checkpoint}")

## Data Collator

The data collator uses Audio Flamingo 3's chat template to format each sample as a conversation: the user provides audio and asks "Describe the audio", and the assistant provides the caption. The processor handles tokenization and label creation.

In [ ]:
class AudioFlamingo3DataCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        conversation = []
        for feature in features:
            sample = [
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": "Describe the audio."},
                        {"type": "audio", "audio": feature["audio"]["array"]},
                    ],
                },
                {
                    "role": "assistant",
                    "content": [{"type": "text", "text": feature["caption"]}],
                },
            ]
            conversation.append(sample)

        return self.processor.apply_chat_template(
            conversation,
            tokenize=True,
            add_generation_prompt=False,
            return_dict=True,
            output_labels=True,
        )

data_collator = AudioFlamingo3DataCollator(processor)

## Full Fine-tuning

We train using `max_steps` since the dataset is streamed. Gradient checkpointing and `save_only_model=True` help manage GPU memory and disk usage.

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./audio-flamingo-3-finetuned",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=5e-5,
    max_steps=50,  # increase for real training (e.g. 1000)
    bf16=True,
    logging_steps=10,
    eval_steps=50,
    save_steps=50,
    save_total_limit=2,
    save_only_model=True,
    eval_strategy="steps",
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,  # must be 0 for streaming datasets
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

trainer.train()

## LoRA Fine-tuning

For parameter-efficient training, we apply LoRA adapters to the LLM's attention and MLP layers. This drastically reduces the number of trainable parameters.

In [ ]:
from peft import LoraConfig, get_peft_model

# Reload model for LoRA training
model_lora = AudioFlamingo3ForConditionalGeneration.from_pretrained(
    model_checkpoint,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model_lora = get_peft_model(model_lora, lora_config)
model_lora.print_trainable_parameters()

In [ ]:
# Re-create streaming dataset iterators for LoRA training
train_dataset_lora = dataset.take(100)
eval_dataset_lora = dataset.skip(100).take(20)

training_args_lora = TrainingArguments(
    output_dir="./audio-flamingo-3-lora-finetuned",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=1e-4,
    max_steps=50,  # increase for real training (e.g. 500)
    bf16=True,
    logging_steps=10,
    eval_steps=50,
    save_steps=50,
    save_total_limit=2,
    save_only_model=True,
    eval_strategy="steps",
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
)

trainer_lora = Trainer(
    model=model_lora,
    args=training_args_lora,
    train_dataset=train_dataset_lora,
    eval_dataset=eval_dataset_lora,
    data_collator=data_collator,
)

trainer_lora.train()

## Inference

Test the fine-tuned model on an audio sample. For LoRA models, load the base model and apply the adapters with `PeftModel`.

In [ ]:
from datasets import load_dataset, Audio

# Load a test sample
test_ds = load_dataset("OpenSound/AudioCaps", split="train", streaming=True)
test_ds = test_ds.cast_column("audio", Audio(sampling_rate=16000))
sample = next(iter(test_ds))

conversation = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Describe the audio."},
            {"type": "audio"},
        ],
    }
]

inputs = processor.apply_chat_template(
    [conversation],
    audio=[sample["audio"]["array"]],
    sampling_rate=16000,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
).to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=200)

caption = processor.batch_decode(
    outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True
)[0]

print(f"Ground truth: {sample['caption']}")
print(f"Generated:    {caption}")